### MotifDiff - Predicted Transcription Factor Binding Alterations 

In [ ]:
# load and zip the GRCh38 build of the human genome as reference 
!wget https://hgdownload.soe.ucsc.edu/goldenPath/hg38/bigZips/hg38.fa.gz
!gunzip hg38.fa.gz

In [ ]:
# install samtools 
!apt-get update
!apt-get install -y samtools

In [ ]:
# index the reference genome using samtools 
!samtools faidx hg38.fa

In [ ]:
# install motifdiff
!pip install MotifDiff-pkg

In [ ]:
# install biopython 
!pip install biopython

In [ ]:
import numpy as np
import pandas as pd

In [ ]:
def load_clean_CSV_to_VCF(df_path, phenotype_classification):
    # load CSV
    df = pd.read_csv(df_path)

    # 1. format chromosome strings
    df['CHROM'] = df['CHROM'].astype(str).apply(lambda x: x if x.startswith('chr') else 'chr' + x)

    # 2. clean fields directly on the original input data
    df['REF'] = df['REF'].astype(str).str.strip().str.upper().apply(lambda x: x.split(',')[0])
    df['ALT'] = df['ALT'].astype(str).str.strip().str.upper().apply(lambda x: x.split(',')[0])

    # 3. filter for valid DNA bases
    dna_bases = set('ACGTN')
    df = df[df['REF'].apply(lambda x: all(c in dna_bases for c in x))]
    df = df[df['ALT'].apply(lambda x: all(c in dna_bases for c in x))]

    # 4. construct the final VCF structure
    vcf_df = pd.DataFrame({
        '#CHROM': df['CHROM'],
        'POS': df['POS'],
        'ID': '.',
        'REF': df['REF'],
        'ALT': df['ALT'],
        'QUAL': '.',
        'FILTER': 'PASS',
        'INFO': '.'
    })

    # 5. save the final clean version directly
    clean_vcf = f'clean_variants_{phenotype_classification}.vcf'
    with open(clean_vcf, 'w') as f:
        f.write('##fileformat=VCFv4.2\n')
        f.write('##reference=hg38\n')
        vcf_df.to_csv(f, sep='\t', index=False)

    print(f"Cleaned VCF saved to {clean_vcf}")
    return clean_vcf

In [ ]:
# load the JASPAR 2024 CORE Vertebrates MEME file
!wget https://jaspar.elixir.no/download/data/2024/CORE/JASPAR2024_CORE_vertebrates_non-redundant_pfms_meme.txt -O motifs.meme

In [ ]:
# check the MEME file structure 
!head -n 20 motifs.meme

In [ ]:
# make a directory for MotifDiff results 
!mkdir -p motifdiff_results

In [ ]:
import inspect
import MotifDiff

print(inspect.getfile(MotifDiff))

In [ ]:
import MotifDiff.MotifDiff as md
print(inspect.getfile(md))

In [ ]:
# corrected max position issue in the original MotifDiff code 

In [ ]:
file_path = "/usr/local/lib/python3.12/dist-packages/MotifDiff/MotifDiff.py"

with open(file_path, "w") as f:
    f.write("""
    # %load /usr/local/lib/python3.12/dist-packages/MotifDiff/MotifDiff.py
#!/usr/bin/env python
from torch.nn import functional as F
import pandas as pd
from .util import vcfData, MEME_probNorm, MEME_FABIAN, MCspline_fitting, mc_spline #
import torch
import numpy as np
from enum import Enum
import typer
import pickle
import time
import resource
import matplotlib.pyplot as plt
#import pyarrow.feather as feather


app = typer.Typer()

torch.backends.cudnn.deterministic = True

def write_output_diff(filename, mat, names, index=None):
    if index is None:
        pd.DataFrame(mat, columns = names).to_csv(filename, header=True, index=False, sep="\t")
        #feather.write_feather(pd.DataFrame(mat, columns = names), f"{filename}.feather")
    else:
        pd.DataFrame(mat, columns = names, index=index).to_csv(filename, header=True, index=True, sep="\t")
        #feather.write_feather(pd.DataFrame(mat, columns = names, index=index), f"{filename}.feather")


class mode_type(str, Enum):
    max = "max"
    average = "average"

class kernel_type(str, Enum):
    TFFM = "TFFM"
    PWM = "PWM"

class norm_type(str, Enum):
    iid = "iid"
    motif_based = "motif_based"
    mixture = "mixture"
    from_mono = "from_mono"


class score_type(str, Enum):
    FABIAN = "FABIAN"
    probNorm = "probNorm"
    NONE = "NONE"

class nucleotide_type(str, Enum):
    mono = "mono"
    di = "di"

class strand_type(str, Enum):
    fwd = "+"
    rev = "-"

# @app.command()
# def variantdiff(genome: str = typer.Option(..., help="fasta file for the genome"),
#                 motif_file: str = typer.Option(..., "--motif", help="meme file for the motifs"),
#                 vcf: str = typer.Option(..., help="vcf file"),
#                 strand: strand_type = typer.Option(None, "--strand", help="whether you want to look on +/- strand of the sequence (default is +)"),
#                 diff_score: score_type = typer.Option(score_type.NONE, "--method", help="how to calculate the diff score (FABIAN/probNorm/NONE)"),
#                 max_scale: bool = typer.Option(False, "--MaxScale", help="Apply max transformation or not"),
#                 nucleotide: nucleotide_type = typer.Option(nucleotide_type.mono, "--nuc", help="length of the nucleotides in the motifs (mono/di)"),
#                 norm_method: norm_type = typer.Option(None, "--methodNorm", help="what method to use for the normalization (iid/motifbased/mixture)"),
#                 normalization_file: str = typer.Option(None, "--norm", help="file including normalization params. should be consistent with the transform option"),
#                 mode: mode_type = typer.Option(mode_type.max, help="Operation mode for the pooling layer (max/average)"),
#                 batch: int = typer.Option(128, help="batch size"),
#                 out_file: str = typer.Option('output', "--out", help="output directory"),
#                 window:int = typer.Option(0, help="window size"), # change this in a way that if it gets a value use that value and if not the default will be kernel size (instead of setting 0 as the default, make it none or not receiving an input as the default)
#                 kernel: kernel_type = typer.Option(kernel_type.PWM, help="Choose between PWM (4 dimensional) or TFFM (16 dimensional) (default = PWM).")
#                 ):

#     s = time.time()
#     if strand is None:
#         strand_specific = False
#         strand = '+'
#     else:
#         strand_specific = True
#         strand = strand

#     kernel = kernel.value
#     if kernel == "PWM":
#         if diff_score == "FABIAN":
#             motif = MEME_FABIAN()
#             print(f"Reading the motifs from {motif_file}")
#             kernels, kernel_mask, kernel_norms = motif.parse(motif_file, nuc=nucleotide, strand_specific=strand_specific)
#         else:
#             if normalization_file is None:
#                 motif = MEME_probNorm()
#                 kernels, kernel_mask = motif.parse(motif_file, nuc=nucleotide, transform=max_scale, strand_specific=strand_specific)
#                 if diff_score=="probNorm":
#                     print("normalization params not given. normalization process running...")
#                     spline_list = MCspline_fitting(kernels, nuc=nucleotide)


#             else:
#                 if "max_scaled" in normalization_file:
#                     max_scale=True
#                 else:
#                     max_scale=False
#                 motif = MEME_probNorm()
#                 kernels, kernel_mask = motif.parse(motif_file, nuc=nucleotide, transform=max_scale)
#                 with open(normalization_file, "rb") as fp:
#                     spline_list = pickle.load(fp)

#         if window==0:
#             windowsize = kernels.shape[2]
#         else:
#             windowsize=window


#     print(f"looking on {strand} strand for motif matchs on windowsize={windowsize}")
#     segments = vcfData(vcf, batch, genome, windowsize, dinucleotide=(nucleotide=="di"), strand=strand)

#     outRef = np.empty((segments.n, motif.nmotifs), dtype=np.float32)
#     outAlt = np.empty((segments.n, motif.nmotifs), dtype=np.float32)
#     refMaxPos = np.empty((segments.n, motif.nmotifs), dtype=np.float32)
#     altMaxPos = np.empty((segments.n, motif.nmotifs), dtype=np.float32)
#     alpha = 0.1
#     print(f"Batch size: {batch}")
#     print("Calculating convolutions...")
#     for i in range(len(segments)):
#         print(f"Batch {i+1}:")
#         i1, i2 = i*batch, (i+1)*batch
#         if i2 >= segments.n: i2 = segments.n
#         matref, maskref, matalt, maskalt = segments[i]
#         #print('before convolution', matref.shape, matref[0,0:5,:]-matalt[0,0:5,:])

#         ref = F.conv1d(matref, kernels)
#         alt = F.conv1d(matalt, kernels)
#         #print('after convolution', ref.shape, ref[0,0:5,:]-alt[0,0:5,:])
#         #print("ref shape after conv:", ref.shape)
#         if window==0:
#             motif_mask = F.conv1d(maskref, kernel_mask)
#             ref[motif_mask == 0] = -torch.inf
#             motif_mask = F.conv1d(maskalt, kernel_mask)
#             alt[motif_mask == 0] = -torch.inf

#         if diff_score == "NONE":
#             ref = ref.view(ref.shape[0], ref.shape[1]//2, ref.shape[2]*2)
#             alt = alt.view(alt.shape[0], alt.shape[1]//2, alt.shape[2]*2)
#             ref = F.max_pool1d(ref, ref.shape[2])
#             alt = F.max_pool1d(alt, alt.shape[2])
#             ref = np.squeeze(ref).numpy()
#             alt = np.squeeze(alt).numpy()

#         if diff_score == "probNorm":
#             #print('before anything', ref.shape, ref[0,0:5,:]-alt[0,0:5,:])
#             if mode == "average":
#                 ref = ref.view(ref.shape[0], ref.shape[1]//2, ref.shape[2]*2)
#                 alt = alt.view(alt.shape[0], alt.shape[1]//2, alt.shape[2]*2)
#                 ref = mc_spline(ref, spline_list)
#                 alt = mc_spline(alt, spline_list)
#                 ref = np.nan_to_num(ref, nan=0)
#                 alt = np.nan_to_num(alt, nan=0)
#                 ref = F.avg_pool1d(torch.tensor(ref), ref.shape[2])
#                 alt = F.avg_pool1d(torch.tensor(alt), alt.shape[2])
#             if mode == "max":
#                 ref = ref.view(ref.shape[0], ref.shape[1]//2, ref.shape[2]*2)
#                 alt = alt.view(alt.shape[0], alt.shape[1]//2, alt.shape[2]*2)
#                 #print(ref[0,0,:])
#                 ref = np.nan_to_num(ref, nan=0)
#                 alt = np.nan_to_num(alt, nan=0)
#                 #print('before max pool', ref.shape)
#                 _,ref_max_pos = torch.max(torch.tensor(ref[:,:,:windowsize]), dim=2)
#                 _,alt_max_pos = torch.max(torch.tensor(alt[:,:,:windowsize]), dim=2)
#                 #print(ref_max_pos, ref_max_pos.shape)
#                 #print(alt_max_pos, alt_max_pos.shape)
#                 ref = F.max_pool1d(torch.tensor(ref), ref.shape[2])
#                 alt = F.max_pool1d(torch.tensor(alt), alt.shape[2])
#                 #print('after max pool', ref.shape, ref[0,0:5,:]-alt[0,0:5,:])

#                 ref = mc_spline(ref, spline_list)
#                 alt = mc_spline(alt, spline_list)
#                 #print('after spline', ref.shape)

#             ref = np.squeeze(ref).numpy()
#             alt = np.squeeze(alt).numpy()


#         if diff_score == "FABIAN":
#             ref = F.max_pool1d(ref, ref.shape[2]).numpy()
#             alt = F.max_pool1d(alt, alt.shape[2]).numpy()
#             ref = np.max(ref.reshape(ref.shape[0],-1,2), axis=2) # separates the convolutions from the original kernel and the reverse complement kernel into two differetn columns AND then keeps the maximum between those two
#             alt = np.max(alt.reshape(alt.shape[0],-1,2), axis=2)


#         if 'ref_max_pos' not in locals():
#           ref_max_pos = np.zeros((i2 - i1, motif.nmotifs))
#           alt_max_pos = np.zeros((i2 - i1, motif.nmotifs))

#         outRef[i1:i2, :motif.nmotifs] = ref
#         outAlt[i1:i2, :motif.nmotifs] = alt
#         refMaxPos[i1:i2, :motif.nmotifs] = ref_max_pos
#         altMaxPos[i1:i2, :motif.nmotifs] = alt_max_pos

#     print("calculating diff...")
#     if diff_score == "FABIAN":
#         outRef = 1 - outRef/kernel_norms[np.newaxis,:]
#         outAlt = 1 - outAlt/kernel_norms[np.newaxis,:]
#         mask = outRef > outAlt
#         f = np.empty_like(outRef)
#         f[np.where(mask)] = -(1-outAlt[mask]+alpha)/(1-outRef[mask]+alpha)+1
#         mask = outAlt >= outRef
#         f[np.where(mask)] = (1-outRef[mask]+alpha)/(1-outAlt[mask]+alpha)-1
#         outDiff = 2/(1+np.power(2, -2*f)) - 1


#     if diff_score == "probNorm" or diff_score == "NONE":
#         outDiff = outAlt-outRef

#     motif_names = motif.names


#     print(f"Writing the results to {out_file}_{nucleotide}_{diff_score}_{mode}")
#     write_output_diff(out_file+f"_{nucleotide}_{diff_score}_{mode}.alt", outAlt, motif_names, segments.names())
#     write_output_diff(out_file+f"_{nucleotide}_{diff_score}_{mode}.ref", outRef, motif_names, segments.names())
#     write_output_diff(out_file+f"_{nucleotide}_{diff_score}_{mode}.diff", outDiff, motif_names, segments.names())

#     write_output_diff(out_file+f"_{nucleotide}_{diff_score}.refMaxPos", refMaxPos, motif_names, segments.names())
#     write_output_diff(out_file+f"_{nucleotide}_{diff_score}.altMaxPos", altMaxPos, motif_names, segments.names())


#     e = time.time()
#     print(f"real runtime for {out_file}_{nucleotide}_{diff_score}_{mode} = ", e-s)

@app.command()
def variantdiff(genome: str = typer.Option(...),
                motif_file: str = typer.Option(..., "--motif"),
                vcf: str = typer.Option(...),
                strand: strand_type = typer.Option(None, "--strand"),
                diff_score: score_type = typer.Option(score_type.NONE, "--method"),
                max_scale: bool = typer.Option(False, "--MaxScale"),
                nucleotide: nucleotide_type = typer.Option(nucleotide_type.mono, "--nuc"),
                norm_method: norm_type = typer.Option(None, "--methodNorm"),
                normalization_file: str = typer.Option(None, "--norm"),
                mode: mode_type = typer.Option(mode_type.max),
                batch: int = typer.Option(128),
                out_file: str = typer.Option('output', "--out"),
                window:int = typer.Option(0),
                kernel: kernel_type = typer.Option(kernel_type.PWM)
                ):

    s = time.time()

    if strand is None:
        strand_specific = False
        strand = '+'
    else:
        strand_specific = True
        strand = strand

    kernel = kernel.value

    if kernel == "PWM":
        if diff_score == "FABIAN":
            motif = MEME_FABIAN()
            kernels, kernel_mask, kernel_norms = motif.parse(motif_file, nuc=nucleotide, strand_specific=strand_specific)
        else:
            if normalization_file is None:
                motif = MEME_probNorm()
                kernels, kernel_mask = motif.parse(motif_file, nuc=nucleotide, transform=max_scale, strand_specific=strand_specific)
                if diff_score=="probNorm":
                    spline_list = MCspline_fitting(kernels, nuc=nucleotide)
            else:
                if "max_scaled" in normalization_file:
                    max_scale=True
                else:
                    max_scale=False
                motif = MEME_probNorm()
                kernels, kernel_mask = motif.parse(motif_file, nuc=nucleotide, transform=max_scale)
                with open(normalization_file, "rb") as fp:
                    spline_list = pickle.load(fp)

        windowsize = kernels.shape[2] if window == 0 else window

    print(f"looking on {strand} strand for motif matchs on windowsize={windowsize}")

    segments = vcfData(vcf, batch, genome, windowsize, dinucleotide=(nucleotide=="di"), strand=strand)

    outRef = np.empty((segments.n, motif.nmotifs), dtype=np.float32)
    outAlt = np.empty((segments.n, motif.nmotifs), dtype=np.float32)
    refMaxPos = np.empty((segments.n, motif.nmotifs), dtype=np.float32)
    altMaxPos = np.empty((segments.n, motif.nmotifs), dtype=np.float32)

    alpha = 0.1

    print(f"Batch size: {batch}")
    print("Calculating convolutions...")

    for i in range(len(segments)):
        print(f"Batch {i+1}:")
        i1, i2 = i*batch, (i+1)*batch
        if i2 >= segments.n:
            i2 = segments.n

        # ALWAYS reset per batch (CRITICAL FIX)
        ref_max_pos = np.zeros((i2 - i1, motif.nmotifs))
        alt_max_pos = np.zeros((i2 - i1, motif.nmotifs))

        matref, maskref, matalt, maskalt = segments[i]

        ref = F.conv1d(matref, kernels)
        alt = F.conv1d(matalt, kernels)

        if window == 0:
            motif_mask = F.conv1d(maskref, kernel_mask)
            ref[motif_mask == 0] = -torch.inf
            motif_mask = F.conv1d(maskalt, kernel_mask)
            alt[motif_mask == 0] = -torch.inf

        if diff_score == "NONE":
            ref = ref.view(ref.shape[0], ref.shape[1]//2, ref.shape[2]*2)
            alt = alt.view(alt.shape[0], alt.shape[1]//2, alt.shape[2]*2)

            ref = F.max_pool1d(ref, ref.shape[2])
            alt = F.max_pool1d(alt, alt.shape[2])

            ref = np.squeeze(ref).numpy()
            alt = np.squeeze(alt).numpy()

        elif diff_score == "probNorm":

            ref = ref.view(ref.shape[0], ref.shape[1]//2, ref.shape[2]*2)
            alt = alt.view(alt.shape[0], alt.shape[1]//2, alt.shape[2]*2)

            ref = np.nan_to_num(ref, nan=0)
            alt = np.nan_to_num(alt, nan=0)

            if mode == "max":
                # SAFE max position extraction
                if ref.shape[2] > 0:
                    w = min(windowsize, ref.shape[2])
                    try:
                        _, ref_max_pos = torch.max(torch.tensor(ref[:, :, :w]), dim=2)
                        _, alt_max_pos = torch.max(torch.tensor(alt[:, :, :w]), dim=2)
                    except Exception as e:
                        print("Max failed:", e)

                ref = F.max_pool1d(torch.tensor(ref), ref.shape[2])
                alt = F.max_pool1d(torch.tensor(alt), alt.shape[2])

                ref = mc_spline(ref, spline_list)
                alt = mc_spline(alt, spline_list)

            elif mode == "average":
                ref = mc_spline(ref, spline_list)
                alt = mc_spline(alt, spline_list)

                ref = F.avg_pool1d(torch.tensor(ref), ref.shape[2])
                alt = F.avg_pool1d(torch.tensor(alt), alt.shape[2])

            ref = np.squeeze(ref).numpy()
            alt = np.squeeze(alt).numpy()

        elif diff_score == "FABIAN":

            ref = F.max_pool1d(ref, ref.shape[2]).numpy()
            alt = F.max_pool1d(alt, alt.shape[2]).numpy()

            ref = np.max(ref.reshape(ref.shape[0], -1, 2), axis=2)
            alt = np.max(alt.reshape(alt.shape[0], -1, 2), axis=2)

        outRef[i1:i2, :motif.nmotifs] = ref
        outAlt[i1:i2, :motif.nmotifs] = alt
        refMaxPos[i1:i2, :motif.nmotifs] = ref_max_pos
        altMaxPos[i1:i2, :motif.nmotifs] = alt_max_pos

    print("calculating diff...")

    if diff_score == "FABIAN":
        outRef = 1 - outRef/kernel_norms[np.newaxis,:]
        outAlt = 1 - outAlt/kernel_norms[np.newaxis,:]

        mask = outRef > outAlt
        f = np.empty_like(outRef)
        f[np.where(mask)] = -(1-outAlt[mask]+alpha)/(1-outRef[mask]+alpha)+1

        mask = outAlt >= outRef
        f[np.where(mask)] = (1-outRef[mask]+alpha)/(1-outAlt[mask]+alpha)-1

        outDiff = 2/(1+np.power(2, -2*f)) - 1

    else:
        outDiff = outAlt - outRef

    motif_names = motif.names

    print(f"Writing the results to {out_file}_{nucleotide}_{diff_score}_{mode}")

    write_output_diff(out_file+f"_{nucleotide}_{diff_score}_{mode}.alt", outAlt, motif_names, segments.names())
    write_output_diff(out_file+f"_{nucleotide}_{diff_score}_{mode}.ref", outRef, motif_names, segments.names())
    write_output_diff(out_file+f"_{nucleotide}_{diff_score}_{mode}.diff", outDiff, motif_names, segments.names())

    write_output_diff(out_file+f"_{nucleotide}_{diff_score}.refMaxPos", refMaxPos, motif_names, segments.names())
    write_output_diff(out_file+f"_{nucleotide}_{diff_score}.altMaxPos", altMaxPos, motif_names, segments.names())

    e = time.time()
    print(f"runtime = {e-s}")


if __name__ == "__main__":
    app()

    """)

print("Saved")

In [ ]:
import sys
import os

# Hardcode the directory path since __file__ doesn't work in Jupyter
sys.path.append('/usr/local/lib/python3.12/dist-packages/MotifDiff')

# Now Python will know exactly where to find 'util'
from util import vcfData, MEME_probNorm, MEME_FABIAN, MCspline_fitting, mc_spline

In [ ]:
# %load /usr/local/lib/python3.12/dist-packages/MotifDiff/MotifDiff.py
#!/usr/bin/env python
from torch.nn import functional as F
import pandas as pd
from util import vcfData, MEME_probNorm, MEME_FABIAN, MCspline_fitting, mc_spline #
import torch
import numpy as np
from enum import Enum
import typer
import pickle
import time
import resource
import matplotlib.pyplot as plt
#import pyarrow.feather as feather


app = typer.Typer()

torch.backends.cudnn.deterministic = True

def write_output_diff(filename, mat, names, index=None):
    if index is None:
        pd.DataFrame(mat, columns = names).to_csv(filename, header=True, index=False, sep="\t")
        #feather.write_feather(pd.DataFrame(mat, columns = names), f"{filename}.feather")
    else:
        pd.DataFrame(mat, columns = names, index=index).to_csv(filename, header=True, index=True, sep="\t")
        #feather.write_feather(pd.DataFrame(mat, columns = names, index=index), f"{filename}.feather")


class mode_type(str, Enum):
    max = "max"
    average = "average"

class kernel_type(str, Enum):
    TFFM = "TFFM"
    PWM = "PWM"

class norm_type(str, Enum):
    iid = "iid"
    motif_based = "motif_based"
    mixture = "mixture"
    from_mono = "from_mono"


class score_type(str, Enum):
    FABIAN = "FABIAN"
    probNorm = "probNorm"
    NONE = "NONE"

class nucleotide_type(str, Enum):
    mono = "mono"
    di = "di"

class strand_type(str, Enum):
    fwd = "+"
    rev = "-"

# @app.command()
# def variantdiff(genome: str = typer.Option(..., help="fasta file for the genome"),
#                 motif_file: str = typer.Option(..., "--motif", help="meme file for the motifs"),
#                 vcf: str = typer.Option(..., help="vcf file"),
#                 strand: strand_type = typer.Option(None, "--strand", help="whether you want to look on +/- strand of the sequence (default is +)"),
#                 diff_score: score_type = typer.Option(score_type.NONE, "--method", help="how to calculate the diff score (FABIAN/probNorm/NONE)"),
#                 max_scale: bool = typer.Option(False, "--MaxScale", help="Apply max transformation or not"),
#                 nucleotide: nucleotide_type = typer.Option(nucleotide_type.mono, "--nuc", help="length of the nucleotides in the motifs (mono/di)"),
#                 norm_method: norm_type = typer.Option(None, "--methodNorm", help="what method to use for the normalization (iid/motifbased/mixture)"),
#                 normalization_file: str = typer.Option(None, "--norm", help="file including normalization params. should be consistent with the transform option"),
#                 mode: mode_type = typer.Option(mode_type.max, help="Operation mode for the pooling layer (max/average)"),
#                 batch: int = typer.Option(128, help="batch size"),
#                 out_file: str = typer.Option('output', "--out", help="output directory"),
#                 window:int = typer.Option(0, help="window size"), # change this in a way that if it gets a value use that value and if not the default will be kernel size (instead of setting 0 as the default, make it none or not receiving an input as the default)
#                 kernel: kernel_type = typer.Option(kernel_type.PWM, help="Choose between PWM (4 dimensional) or TFFM (16 dimensional) (default = PWM).")
#                 ):

#     s = time.time()
#     if strand is None:
#         strand_specific = False
#         strand = '+'
#     else:
#         strand_specific = True
#         strand = strand

#     kernel = kernel.value
#     if kernel == "PWM":
#         if diff_score == "FABIAN":
#             motif = MEME_FABIAN()
#             print(f"Reading the motifs from {motif_file}")
#             kernels, kernel_mask, kernel_norms = motif.parse(motif_file, nuc=nucleotide, strand_specific=strand_specific)
#         else:
#             if normalization_file is None:
#                 motif = MEME_probNorm()
#                 kernels, kernel_mask = motif.parse(motif_file, nuc=nucleotide, transform=max_scale, strand_specific=strand_specific)
#                 if diff_score=="probNorm":
#                     print("normalization params not given. normalization process running...")
#                     spline_list = MCspline_fitting(kernels, nuc=nucleotide)


#             else:
#                 if "max_scaled" in normalization_file:
#                     max_scale=True
#                 else:
#                     max_scale=False
#                 motif = MEME_probNorm()
#                 kernels, kernel_mask = motif.parse(motif_file, nuc=nucleotide, transform=max_scale)
#                 with open(normalization_file, "rb") as fp:
#                     spline_list = pickle.load(fp)

#         if window==0:
#             windowsize = kernels.shape[2]
#         else:
#             windowsize=window


#     print(f"looking on {strand} strand for motif matchs on windowsize={windowsize}")
#     segments = vcfData(vcf, batch, genome, windowsize, dinucleotide=(nucleotide=="di"), strand=strand)

#     outRef = np.empty((segments.n, motif.nmotifs), dtype=np.float32)
#     outAlt = np.empty((segments.n, motif.nmotifs), dtype=np.float32)
#     refMaxPos = np.empty((segments.n, motif.nmotifs), dtype=np.float32)
#     altMaxPos = np.empty((segments.n, motif.nmotifs), dtype=np.float32)
#     alpha = 0.1
#     print(f"Batch size: {batch}")
#     print("Calculating convolutions...")
#     for i in range(len(segments)):
#         print(f"Batch {i+1}:")
#         i1, i2 = i*batch, (i+1)*batch
#         if i2 >= segments.n: i2 = segments.n
#         matref, maskref, matalt, maskalt = segments[i]
#         #print('before convolution', matref.shape, matref[0,0:5,:]-matalt[0,0:5,:])

#         ref = F.conv1d(matref, kernels)
#         alt = F.conv1d(matalt, kernels)
#         #print('after convolution', ref.shape, ref[0,0:5,:]-alt[0,0:5,:])
#         #print("ref shape after conv:", ref.shape)
#         if window==0:
#             motif_mask = F.conv1d(maskref, kernel_mask)
#             ref[motif_mask == 0] = -torch.inf
#             motif_mask = F.conv1d(maskalt, kernel_mask)
#             alt[motif_mask == 0] = -torch.inf

#         if diff_score == "NONE":
#             ref = ref.view(ref.shape[0], ref.shape[1]//2, ref.shape[2]*2)
#             alt = alt.view(alt.shape[0], alt.shape[1]//2, alt.shape[2]*2)
#             ref = F.max_pool1d(ref, ref.shape[2])
#             alt = F.max_pool1d(alt, alt.shape[2])
#             ref = np.squeeze(ref).numpy()
#             alt = np.squeeze(alt).numpy()

#         if diff_score == "probNorm":
#             #print('before anything', ref.shape, ref[0,0:5,:]-alt[0,0:5,:])
#             if mode == "average":
#                 ref = ref.view(ref.shape[0], ref.shape[1]//2, ref.shape[2]*2)
#                 alt = alt.view(alt.shape[0], alt.shape[1]//2, alt.shape[2]*2)
#                 ref = mc_spline(ref, spline_list)
#                 alt = mc_spline(alt, spline_list)
#                 ref = np.nan_to_num(ref, nan=0)
#                 alt = np.nan_to_num(alt, nan=0)
#                 ref = F.avg_pool1d(torch.tensor(ref), ref.shape[2])
#                 alt = F.avg_pool1d(torch.tensor(alt), alt.shape[2])
#             if mode == "max":
#                 ref = ref.view(ref.shape[0], ref.shape[1]//2, ref.shape[2]*2)
#                 alt = alt.view(alt.shape[0], alt.shape[1]//2, alt.shape[2]*2)
#                 #print(ref[0,0,:])
#                 ref = np.nan_to_num(ref, nan=0)
#                 alt = np.nan_to_num(alt, nan=0)
#                 #print('before max pool', ref.shape)
#                 _,ref_max_pos = torch.max(torch.tensor(ref[:,:,:windowsize]), dim=2)
#                 _,alt_max_pos = torch.max(torch.tensor(alt[:,:,:windowsize]), dim=2)
#                 #print(ref_max_pos, ref_max_pos.shape)
#                 #print(alt_max_pos, alt_max_pos.shape)
#                 ref = F.max_pool1d(torch.tensor(ref), ref.shape[2])
#                 alt = F.max_pool1d(torch.tensor(alt), alt.shape[2])
#                 #print('after max pool', ref.shape, ref[0,0:5,:]-alt[0,0:5,:])

#                 ref = mc_spline(ref, spline_list)
#                 alt = mc_spline(alt, spline_list)
#                 #print('after spline', ref.shape)

#             ref = np.squeeze(ref).numpy()
#             alt = np.squeeze(alt).numpy()


#         if diff_score == "FABIAN":
#             ref = F.max_pool1d(ref, ref.shape[2]).numpy()
#             alt = F.max_pool1d(alt, alt.shape[2]).numpy()
#             ref = np.max(ref.reshape(ref.shape[0],-1,2), axis=2) # separates the convolutions from the original kernel and the reverse complement kernel into two differetn columns AND then keeps the maximum between those two
#             alt = np.max(alt.reshape(alt.shape[0],-1,2), axis=2)


#         if 'ref_max_pos' not in locals():
#           ref_max_pos = np.zeros((i2 - i1, motif.nmotifs))
#           alt_max_pos = np.zeros((i2 - i1, motif.nmotifs))

#         outRef[i1:i2, :motif.nmotifs] = ref
#         outAlt[i1:i2, :motif.nmotifs] = alt
#         refMaxPos[i1:i2, :motif.nmotifs] = ref_max_pos
#         altMaxPos[i1:i2, :motif.nmotifs] = alt_max_pos

#     print("calculating diff...")
#     if diff_score == "FABIAN":
#         outRef = 1 - outRef/kernel_norms[np.newaxis,:]
#         outAlt = 1 - outAlt/kernel_norms[np.newaxis,:]
#         mask = outRef > outAlt
#         f = np.empty_like(outRef)
#         f[np.where(mask)] = -(1-outAlt[mask]+alpha)/(1-outRef[mask]+alpha)+1
#         mask = outAlt >= outRef
#         f[np.where(mask)] = (1-outRef[mask]+alpha)/(1-outAlt[mask]+alpha)-1
#         outDiff = 2/(1+np.power(2, -2*f)) - 1


#     if diff_score == "probNorm" or diff_score == "NONE":
#         outDiff = outAlt-outRef

#     motif_names = motif.names


#     print(f"Writing the results to {out_file}_{nucleotide}_{diff_score}_{mode}")
#     write_output_diff(out_file+f"_{nucleotide}_{diff_score}_{mode}.alt", outAlt, motif_names, segments.names())
#     write_output_diff(out_file+f"_{nucleotide}_{diff_score}_{mode}.ref", outRef, motif_names, segments.names())
#     write_output_diff(out_file+f"_{nucleotide}_{diff_score}_{mode}.diff", outDiff, motif_names, segments.names())

#     write_output_diff(out_file+f"_{nucleotide}_{diff_score}.refMaxPos", refMaxPos, motif_names, segments.names())
#     write_output_diff(out_file+f"_{nucleotide}_{diff_score}.altMaxPos", altMaxPos, motif_names, segments.names())


#     e = time.time()
#     print(f"real runtime for {out_file}_{nucleotide}_{diff_score}_{mode} = ", e-s)

@app.command()
def variantdiff(genome: str = typer.Option(...),
                motif_file: str = typer.Option(..., "--motif"),
                vcf: str = typer.Option(...),
                strand: strand_type = typer.Option(None, "--strand"),
                diff_score: score_type = typer.Option(score_type.NONE, "--method"),
                max_scale: bool = typer.Option(False, "--MaxScale"),
                nucleotide: nucleotide_type = typer.Option(nucleotide_type.mono, "--nuc"),
                norm_method: norm_type = typer.Option(None, "--methodNorm"),
                normalization_file: str = typer.Option(None, "--norm"),
                mode: mode_type = typer.Option(mode_type.max),
                batch: int = typer.Option(128),
                out_file: str = typer.Option('output', "--out"),
                window:int = typer.Option(0),
                kernel: kernel_type = typer.Option(kernel_type.PWM)
                ):

    s = time.time()

    if strand is None:
        strand_specific = False
        strand = '+'
    else:
        strand_specific = True
        strand = strand

    kernel = kernel.value

    if kernel == "PWM":
        if diff_score == "FABIAN":
            motif = MEME_FABIAN()
            kernels, kernel_mask, kernel_norms = motif.parse(motif_file, nuc=nucleotide, strand_specific=strand_specific)
        else:
            if normalization_file is None:
                motif = MEME_probNorm()
                kernels, kernel_mask = motif.parse(motif_file, nuc=nucleotide, transform=max_scale, strand_specific=strand_specific)
                if diff_score=="probNorm":
                    spline_list = MCspline_fitting(kernels, nuc=nucleotide)
            else:
                if "max_scaled" in normalization_file:
                    max_scale=True
                else:
                    max_scale=False
                motif = MEME_probNorm()
                kernels, kernel_mask = motif.parse(motif_file, nuc=nucleotide, transform=max_scale)
                with open(normalization_file, "rb") as fp:
                    spline_list = pickle.load(fp)

        windowsize = kernels.shape[2] if window == 0 else window

    print(f"looking on {strand} strand for motif matchs on windowsize={windowsize}")

    segments = vcfData(vcf, batch, genome, windowsize, dinucleotide=(nucleotide=="di"), strand=strand)

    outRef = np.empty((segments.n, motif.nmotifs), dtype=np.float32)
    outAlt = np.empty((segments.n, motif.nmotifs), dtype=np.float32)
    refMaxPos = np.empty((segments.n, motif.nmotifs), dtype=np.float32)
    altMaxPos = np.empty((segments.n, motif.nmotifs), dtype=np.float32)

    alpha = 0.1

    print(f"Batch size: {batch}")
    print("Calculating convolutions...")

    for i in range(len(segments)):
        print(f"Batch {i+1}:")
        i1, i2 = i*batch, (i+1)*batch
        if i2 >= segments.n:
            i2 = segments.n

        # ALWAYS reset per batch (CRITICAL FIX)
        ref_max_pos = np.zeros((i2 - i1, motif.nmotifs))
        alt_max_pos = np.zeros((i2 - i1, motif.nmotifs))

        matref, maskref, matalt, maskalt = segments[i]

        ref = F.conv1d(matref, kernels)
        alt = F.conv1d(matalt, kernels)

        if window == 0:
            motif_mask = F.conv1d(maskref, kernel_mask)
            ref[motif_mask == 0] = -torch.inf
            motif_mask = F.conv1d(maskalt, kernel_mask)
            alt[motif_mask == 0] = -torch.inf

        if diff_score == "NONE":
            ref = ref.view(ref.shape[0], ref.shape[1]//2, ref.shape[2]*2)
            alt = alt.view(alt.shape[0], alt.shape[1]//2, alt.shape[2]*2)

            ref = F.max_pool1d(ref, ref.shape[2])
            alt = F.max_pool1d(alt, alt.shape[2])

            ref = np.squeeze(ref).numpy()
            alt = np.squeeze(alt).numpy()

        elif diff_score == "probNorm":

            ref = ref.view(ref.shape[0], ref.shape[1]//2, ref.shape[2]*2)
            alt = alt.view(alt.shape[0], alt.shape[1]//2, alt.shape[2]*2)

            ref = np.nan_to_num(ref, nan=0)
            alt = np.nan_to_num(alt, nan=0)

            if mode == "max":
                # SAFE max position extraction
                if ref.shape[2] > 0:
                    w = min(windowsize, ref.shape[2])
                    try:
                        _, ref_max_pos = torch.max(torch.tensor(ref[:, :, :w]), dim=2)
                        _, alt_max_pos = torch.max(torch.tensor(alt[:, :, :w]), dim=2)
                    except Exception as e:
                        print("Max failed:", e)

                ref = F.max_pool1d(torch.tensor(ref), ref.shape[2])
                alt = F.max_pool1d(torch.tensor(alt), alt.shape[2])

                ref = mc_spline(ref, spline_list)
                alt = mc_spline(alt, spline_list)

            elif mode == "average":
                ref = mc_spline(ref, spline_list)
                alt = mc_spline(alt, spline_list)

                ref = F.avg_pool1d(torch.tensor(ref), ref.shape[2])
                alt = F.avg_pool1d(torch.tensor(alt), alt.shape[2])

            ref = np.squeeze(ref).numpy()
            alt = np.squeeze(alt).numpy()

        elif diff_score == "FABIAN":

            ref = F.max_pool1d(ref, ref.shape[2]).numpy()
            alt = F.max_pool1d(alt, alt.shape[2]).numpy()

            ref = np.max(ref.reshape(ref.shape[0], -1, 2), axis=2)
            alt = np.max(alt.reshape(alt.shape[0], -1, 2), axis=2)

        outRef[i1:i2, :motif.nmotifs] = ref
        outAlt[i1:i2, :motif.nmotifs] = alt
        refMaxPos[i1:i2, :motif.nmotifs] = ref_max_pos
        altMaxPos[i1:i2, :motif.nmotifs] = alt_max_pos

    print("calculating diff...")

    if diff_score == "FABIAN":
        outRef = 1 - outRef/kernel_norms[np.newaxis,:]
        outAlt = 1 - outAlt/kernel_norms[np.newaxis,:]

        mask = outRef > outAlt
        f = np.empty_like(outRef)
        f[np.where(mask)] = -(1-outAlt[mask]+alpha)/(1-outRef[mask]+alpha)+1

        mask = outAlt >= outRef
        f[np.where(mask)] = (1-outRef[mask]+alpha)/(1-outAlt[mask]+alpha)-1

        outDiff = 2/(1+np.power(2, -2*f)) - 1

    else:
        outDiff = outAlt - outRef

    motif_names = motif.names

    print(f"Writing the results to {out_file}_{nucleotide}_{diff_score}_{mode}")

    write_output_diff(out_file+f"_{nucleotide}_{diff_score}_{mode}.alt", outAlt, motif_names, segments.names())
    write_output_diff(out_file+f"_{nucleotide}_{diff_score}_{mode}.ref", outRef, motif_names, segments.names())
    write_output_diff(out_file+f"_{nucleotide}_{diff_score}_{mode}.diff", outDiff, motif_names, segments.names())

    write_output_diff(out_file+f"_{nucleotide}_{diff_score}.refMaxPos", refMaxPos, motif_names, segments.names())
    write_output_diff(out_file+f"_{nucleotide}_{diff_score}.altMaxPos", altMaxPos, motif_names, segments.names())

    e = time.time()
    print(f"runtime = {e-s}")


# if __name__ == "__main__":
#     app()


In [ ]:
import inspect
import MotifDiff.MotifDiff as md

print(inspect.getsource(md.variantdiff)[:500])

In [ ]:
# run MotifDiff 
!getDiff --genome hg38.fa \
         --vcf clean_variants_high.vcf \
         --motif motifs.meme \
         --out ./motifdiff_results/


In [ ]:
# alternate way to run MotifDiff 

if __name__ == "__main__":
    app(args=[
        "--genome", "hg38.fa",
        "--vcf", "clean_variants_low.vcf",
        "--motif", "motifs.meme",
        "--out", "./motifdiff_results/"
    ])

In [ ]:
# go to the results direction 
!cd motifdiff_results/ && ls -al

In [ ]:
# parse motifdiff .diff file (delta scores based on REF and ALT sequences)
diff_file = "/content/motifdiff_results/_nucleotide_type.mono_score_type.NONE_mode_type.max.diff"
df_diff = pd.read_csv(diff_file, sep="\t")

df_diff.head()